# Multi AI Agent workflow

- Team of AI agents that can work together to achieve a common goal.
- **Reasercher Agent**: Responsible for gathering and analyzing data relevant to the task. Agent will use tavil to search for information, summarize findings, and provide insights to the team.
- **Analyst Agent**: Responsible for interpreting the data provided by the Researcher Agent. This agent will identify patterns, trends, and potential solutions based on the research findings.
- **Writer Agent**: Responsible for documenting the findings and solutions proposed by the Analyst Agent. This agent will create reports, presentations, and other written materials to communicate the team's work effectively.
- **Manager function**: Oversees the workflow of the AI agents, ensuring that tasks are completed efficiently and effectively. The manager will coordinate the efforts of the Researcher, Analyst, and Writer agents, providing guidance and support as needed.

### Setup tools and libraries

In [2]:
!pip install -q openai-agents==0.2.2 python-dotenv requests pydantic

In [18]:
import os
import requests
from IPython.display import display, Markdown
from openai import OpenAI
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, SQLiteSession
from typing_extensions import TypedDict
from pydantic import BaseModel

In [19]:
# Setup env. and Load API keys from .env file
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

if not openai_api_key:
    from getpass import getpass
    openai_api_key = getpass("OpenAI API key (will not be echoed): ")
    
# Ensure the agents/OpenAI client can read the key via the environment variable
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

if not tavily_api_key:
    from getpass import getpass
    tavily_api_key = getpass("Tavily API key (will not be echoed): ")

# Ensure the agents/OpenAI client can read the key via the environment variable
if tavily_api_key:
    os.environ["TAVILY_API_KEY"] = tavily_api_key

openai_client = OpenAI(api_key=openai_api_key)

print("✅ API keys loaded")
print(f"  OpenAI … {openai_api_key[:5]}***")
print(f"  Tavily … {tavily_api_key[:5]}***")

In [17]:
gpt_model = "gpt-6-luna"

### Setup Tavily Search Tool and Markdown class

In [20]:
# Define a function to display markdown in Jupyter Notebook
def print_markdown(text):
    display(Markdown(text))

In [21]:
# A TypedDict describes the expected keys and value types for a dict.
# Here: a search "query" (string) and "max_results" (int).
class TavilySearchParams(TypedDict):
    query: str
    max_results: int

In [22]:
# This decorator (from OpenAI Agents SDK) registers the function as a tool
# that an agent can call. 

@function_tool
def tavily_search(params: TavilySearchParams) -> str:

    # Tavily search endpoint
    url = "https://api.tavily.com/search"

    # Tell the API we're sending JSON
    headers = {"Content-Type": "application/json"}

    # Build the request body:
    # api_key: your Tavily API key (assumed defined elsewhere as tavily_api_key)
    # query: taken from the params dict
    # max_results: use provided value or default to 3 if missing
    payload = {
        "api_key": tavily_api_key,
        "query": params["query"],
        "max_results": params.get("max_results", 3),
    }

    # Send the POST request with JSON body and headers
    response = requests.post(url, json = payload, headers = headers)
    if response.status_code == 200:
        results = response.json().get("results", [])
        summary = "\n".join([f"- {r['title']}: {r['content']}" for r in results])
        print_markdown(summary if summary else "No relevant results found.")
        return summary if summary else "No relevant results found."
    else:
        return f"Tavily API error: {response.status_code}"

print("✅ Tavily search tool ready.")

### Researcher AI agent - Define and Execute

In [23]:
# Let's define our Researcher AI Agent

# Define a Pydantic model for the agent's output.
# This ensures the agent's final output will have a single field: "summary" (string).
# Note that this will be used in both the researcher and analyst agents
class AnalysisSummary(BaseModel):
    summary: str

In [24]:
# Create an AI agent called "Researcher"
researcher_agent = Agent(
    name = "Researcher",
    instructions = """
        ## Context
        You are a research agent with access to the Tavily search tool.

        ## Instruction
        Given a user query, use the Tavily search tool to find relevant information and summarize the key findings.

        ## Input
        - A research query from the user.

        ## Output
        - A summary of key findings in a maximum of 5 bullet points.
        """,
    model = gpt_model,
    tools = [tavily_search],
    output_type = AnalysisSummary
)

print("✅ Researcher AI agent is now ready")

In [25]:
# Let's run the researcher AI agent
researcher_result = await Runner.run(researcher_agent, "Why is Little Mermaid so popular in Denmark and what is the history behind it?")
researcher_result

In [26]:
print_markdown(f"### 🤖 Agent's Answer\n{researcher_result.final_output.summary}")

### Analyst AI agent - Define and Execute 

In [27]:
# Let's define our Analyst AI Agent
analyst_agent = Agent(
    name = "Analyst",
    instructions = """
        ## Context
        You are an analyst who receives research notes generated by the research agent.

        ## Instruction
        Given the research notes, analyze the content and extract key trends, risks, or insights.

        ## Input
        - Research notes (summaries of findings from the research agent).

        ## Output
        - A concise analysis (no more than 2 paragraphs) highlighting key trends, risks, or insights.
        """,
    model = gpt_model,
    output_type = AnalysisSummary,
)

print("✅ Analyst AI agent is now ready")

### Writer AI agent - Define and Execute 
**Finale Executive Summary Report Generation**

In [28]:
# Define a Pydantic model for the agent's output
# This specifies exactly what fields the Writer agent should return
class FinalReport(BaseModel):
    short_summary: str                  # A brief 2–3 sentence executive summary
    markdown_report: str                # A detailed report in Markdown format (at least 500 words)
    follow_up_questions: list[str]      # A list of 3–5 suggested follow-up research questions

In [29]:
# Create an AI agent called "Writer"
writer_agent = Agent(
    name = "Writer",
    instructions = """
        You are a senior market analyst tasked with generating an executive-level research report.

        ## Context
        You will receive:
        - The original user query (the research question or topic of interest).
        - Summaries and analyses generated by helper tools: 'researcher_agent' (for bullet-point research findings) and 'analyst_agent' (for key trends, risks, or insights).

        You may call these helper tools as needed to gather additional information or clarify findings.

        ## Instructions
        Your job is to synthesize all available information and produce the following outputs:
        1. **Executive Summary:** Write a concise summary (2-3 sentences) highlighting the most important findings relevant to the original query.
        2. **Detailed Markdown Report:** Compose a comprehensive, well-structured report in markdown format (at least 500 words) that covers key findings, context, implications, and supporting evidence.
        3. **Follow-up Research Questions:** Suggest 3-5 thoughtful follow-up questions for further investigation.

        ## Input
        - The original user query.
        - Summaries and analyses from the helper tools.

        ## Output
        Return a structured object with:
        - `short_summary`: The executive summary.
        - `markdown_report`: The detailed markdown report.
        - `follow_up_questions`: A list of 3-5 follow-up research questions.
        """,
    model = gpt_model,
    output_type = FinalReport,
)

print("✅ Writer AI Agent is now ready")

### Build Manager - coordinate workflow of AI agents
**Orchestrate the full pipeline - workflow of the Researcher, Analyst, and Writer agents**

In [31]:
# Create a session to store and share information between agents
session = SQLiteSession("research_agent_practice")

# Define an async function that takes the user's question and runs the agents
async def manager_run(user_query: str):
    # Show the user’s question in a nice markdown format
    print_markdown(f"**User's Request:** {user_query}")

    # Step 1: Run the researcher agent to gather information about the user's query
    researcher_result = await Runner.run(researcher_agent, user_query, session = session)
    research_summary = researcher_result.final_output.summary  # Get the short research summary

    # Step 2: Run the analyst agent to analyze the research findings
    analyst_result = await Runner.run(analyst_agent, research_summary, session = session)
    analysis_summary = analyst_result.final_output.summary  # Get the short analysis summary

    # Step 3: Prepare the combined input for the writer agent
    input_for_writer = (
        f"Original query: {user_query}\n"
        f"Research summary: {research_summary}\n"
        f"Analysis summary: {analysis_summary}"
    )

    # Step 4: Run the writer agent to create the final report
    writer_result = await Runner.run(writer_agent, input_for_writer, session = session)
    final_output: FinalReport = writer_result.final_output  # The final report from the writer

    # Step 5: Display the results in a nice, clear format
    print_markdown("---")
    print_markdown(f"### 📝 **Short Summary:**\n{final_output.short_summary}")  # Quick overview
    print_markdown("\n\n-----------------\n\n")
    print_markdown(f"### 📄 **Full Report (Markdown):**\n{final_output.markdown_report}")  # Detailed report
    print_markdown("\n\n-----------------\n\n")
    print_markdown(
        "### 🔍 **Follow-Up Questions:**\n- "
        + "\n- ".join(final_output.follow_up_questions)  # Any extra questions the user might explore
    )
    print_markdown("\n\n-----------------\n\n")

In [32]:
await manager_run("What is the public sentiment and expert reviews about latest iPhone?")

## Writer AI Agent - (Diff. language)
**Writer function that generate output in different language**

**PRACTICE OPPORTUNITY:**
- **Rewrite the `writer` agent’s instructions so that it generates all outputs entirely in different language (Executive Summary, Detailed Markdown Report, and Follow-up Research Questions).**
- **Note: We will test this AI Agent in the next task so stay tuned!**

In [ ]:
## Variation to Report generation.
## **Rewrite the `writer` agent’s instructions so that it generates all outputs entirely in Gujarati (Executive Summary, Detailed Markdown Report, and Follow-up Research Questions).**

writer_agent = Agent(
    name = "Writer",
    instructions = """
You are a senior market analyst tasked with generating an executive-level research report **in Gujarati**.

## Context
You will receive:
- The original user query (the research question or topic of interest).
- Summaries and analyses generated by helper tools: 'researcher_agent' (for bullet-point research findings) and 'analyst_agent' (for key trends, risks, or insights).

You may call these helper tools as needed to gather additional information or clarify findings.

## Instructions
Your job is to synthesize all available information and produce the following outputs in **Gujarati**:
1. **Executive Summary:** Write a concise summary (2-3 sentences) in Gujarati highlighting the most important findings relevant to the original query.
2. **Detailed Markdown Report:** Compose a comprehensive, well-structured report in Gujarati in markdown format (at least 500 words) that covers key findings, context, implications, and supporting evidence.
3. **Follow-up Research Questions:** Suggest 3-5 thoughtful follow-up questions in Gujarati for further investigation.

## Input
- The original user query.
- Summaries and analyses from the helper tools.

## Output
Return a structured object with:
- `short_summary`: The executive summary in Gujarati.
- `markdown_report`: The detailed markdown report in Gujarati.
- `follow_up_questions`: A list of 3-5 follow-up research questions in Gujarati.
""",
    model = "gpt-4.1-mini",
    output_type = FinalReport)


In [ ]:
await manager_run("What is the public sentiment and expert reviews about latest iPhone?")

## PRACTICE SOLUTION

- **Use the `researcher_agent` and `analyst_agent` to analyze a specific research question or topic.**

  - **Provide a research question (e.g., "What are the latest trends in Batteries for Electric Vehicles?").**
  - **Call the `researcher_agent` with the research question to obtain key findings.**
  - **Pass the researcher's summary to the `analyst_agent` to extract key trends, risks, or insights.**
  - **Show how you would structure the conversation and pass information between the agents.**


In [33]:
# Let's run the researcher AI agent
researcher_result = await Runner.run(researcher_agent, "Electric vehicle batteries trends in 2026 and beyond")
print_markdown(f"### 🤖 Researcher Agent's Reaserach\n{researcher_result.final_output.summary}")

In [34]:
analyst_result = await Runner.run(analyst_agent, researcher_result.final_output.summary)
print_markdown(f"### 🤖 Analyst Agent's Answer\n{analyst_result.final_output.summary}")

## PRACTICE SOLUTION - Part 2

- **Create a new multi-agent team that performs creative advertising:**
  - **Define a `Creative_Director` agent that brainstorms 3–5 ad ideas.**
  - **Define a `Strategist` agent that selects the top 2 ideas and explains the reasoning.**
  - **Define a `Copywriter` agent that writes tweets for each top campaign.**
  - **Build a pipeline where the output of each agent is passed to the next, similar to the previous example.**
  - **Try running the full pipeline on a sample prompt (e.g., "Launch campaign for a new eco-friendly water bottle in Bali").**

In [36]:
session = SQLiteSession("creative_pipeline_practice")

class CreativeIdeas(BaseModel):
    ideas: list[str]

creative_director = Agent(
    name = "Creative_Director",
    instructions = """
        Context:
        You are a creative director at an advertising agency.

        Instruction:
        Brainstorm 3-5 creative ad campaign ideas for the given product.

        Input:
        The product name or description.

        Output:
        A list of 3-5 creative campaign ideas (as bullet points or short phrases).
        """,
    model = gpt_model,
    output_type = CreativeIdeas
)

In [37]:

class SelectedCampaigns(BaseModel):
    top_campaigns: list[str]
    reasoning: str

strategist = Agent(
    name = "Strategist",
    instructions = """
        Context:
        You are a marketing strategist evaluating campaign ideas.

        Instruction:
        From the provided list of campaign ideas, select the top 2 that are most promising and explain your reasoning.

        Input:
        A list of campaign ideas (one per line).

        Output:
        A list of the top 2 campaign ideas and a paragraph explaining why they stand out.
        """,
    model = gpt_model,
    output_type = SelectedCampaigns,
)


In [38]:
class TweetCopy(BaseModel):
    tweets: list[str]

copywriter = Agent(
    name = "Copywriter",
    instructions = """
        Context:
        You are a copywriter creating social media content.

        Instruction:
        For each of the top campaign ideas provided, write 2-3 engaging tweets that could be used to promote the campaign.

        Input:
        The selected top campaign ideas (one per line).

        Output:
        A list of 2-3 tweets for each campaign idea.
        """,
    model = gpt_model,
    output_type = TweetCopy)

In [39]:
async def creative_pipeline(product_name: str):
    director_result = await Runner.run(creative_director, product_name, session = session)
    ideas = director_result.final_output.ideas

    strategist_input = "\n".join(ideas)
    strategist_result = await Runner.run(strategist, strategist_input, session = session)
    top_campaigns = strategist_result.final_output.top_campaigns

    tweets_input = "\n".join(top_campaigns)
    copywriter_result = await Runner.run(copywriter, tweets_input, session = session)
    tweets = copywriter_result.final_output.tweets

    print_markdown("### **Top Campaigns:**\n- " + "\n- ".join(top_campaigns))
    print_markdown("### **Tweets:**\n- " + "\n- ".join(tweets))

In [40]:
await creative_pipeline("Launch campaign for a new eco-friendly water bottle in Bali")